In [ ]:
import requests
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

spacex_url = "https://api.spacexdata.com/v4/launches/past"
rocket_url = "https://api.spacexdata.com/v4/rockets/"
payload_url = "https://api.spacexdata.com/v4/payloads/"
launchpad_url = "https://api.spacexdata.com/v4/launchpads/"
cores_url = "https://api.spacexdata.com/v4/cores/"

launches = requests.get(spacex_url).json()

spacex_data = []

for launch in launches:
    if launch["rocket"] != "5e9d0d95eda69973a809d1ec":  # Falcon 9 only
        continue

    payloads = launch.get("payloads", [])
    cores = launch.get("cores", [])

    payload_mass = None
    orbit = None
    if payloads:
        payload_info = requests.get(payload_url + payloads[0]).json()
        payload_mass = payload_info.get("mass_kg", None)
        orbit = payload_info.get("orbit", None)

    launchpad_info = requests.get(launchpad_url + launch["launchpad"]).json()
    launch_site = launchpad_info.get("name", None)
    latitude = launchpad_info.get("latitude", None)
    longitude = launchpad_info.get("longitude", None)

    core_serial = None
    landing_success = None
    reused = None
    landing_type = None
    flight = None
    block = None
    gridfins = None
    legs = None

    if cores:
        core = cores[0]
        core_serial = core.get("core", None)
        landing_success = core.get("landing_success", None)
        reused = core.get("reused", None)
        landing_type = core.get("landing_type", None)
        flight = core.get("flight", None)
        gridfins = core.get("gridfins", None)
        legs = core.get("legs", None)

        if core_serial:
            core_info = requests.get(cores_url + core_serial).json()
            block = core_info.get("block", None)
            core_serial = core_info.get("serial", None)

    spacex_data.append({
        "FlightNumber": launch.get("flight_number"),
        "Date": launch.get("date_utc")[:10],
        "BoosterVersion": "Falcon 9",
        "PayloadMass": payload_mass,
        "Orbit": orbit,
        "LaunchSite": launch_site,
        "Outcome": landing_type,
        "Flights": flight,
        "GridFins": gridfins,
        "Reused": reused,
        "Legs": legs,
        "LandingPad": None,
        "Block": block,
        "ReusedCount": None,
        "Serial": core_serial,
        "Longitude": longitude,
        "Latitude": latitude,
        "Class": 1 if landing_success is True else 0
    })

df = pd.DataFrame(spacex_data)
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,6,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None,1,False,False,False,None,1,None,B0003,-80.577366,28.561857,0
1,7,2010-12-08,Falcon 9,NaN,LEO,CCSFS SLC 40,None,1,False,False,False,None,1,None,B0004,-80.577366,28.561857,0
2,8,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None,1,False,False,False,None,1,None,B0005,-80.577366,28.561857,0
3,9,2012-10-08,Falcon 9,400.0,ISS,CCSFS SLC 40,None,1,False,False,False,None,1,None,B0006,-80.577366,28.561857,0
4,10,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None,1,False,False,False,None,1,None,B0007,-80.577366,28.561857,0


In [ ]:
df.to_csv("dataset_part_1.csv", index=False)